In [1]:
import os
import sys
from pathlib import Path

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("otto-baselines")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

from src.baselines import build_covisit_buys, popular_items, recommend
from src.metrics import recall_at_20

print("ready:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/25 11:32:35 WARN Utils: Your hostname, Omkars-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.75 instead (on interface en0)
26/11/25 11:32:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 11:32:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 11:32:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


ready: 4.3.0


In [2]:
from pyspark.sql import functions as F

tr = spark.read.parquet(str(DATA_PROCESSED / "split_train"))
ti = spark.read.parquet(str(DATA_PROCESSED / "test_input"))
tl = spark.read.parquet(str(DATA_PROCESSED / "test_labels"))

max_ts = spark.read.parquet(str(DATA_PROCESSED / "train_events")).agg(F.max("ts")).first()[0]
cutoff = max_ts - 7 * 24 * 3600 + 1

print("cutoff:", cutoff)
print("train events:", tr.count())
print("test sessions:", tl.count())

cutoff: 1661119200
train events: 163955181
test sessions: 1801251


In [3]:
popular = popular_items(tr, cutoff)
cv = build_covisit_buys(tr)
sessions = tl.select("session")

results = {
    "popular only": recall_at_20(recommend(ti, sessions, popular, use_history=False), tl),
    "history + popular": recall_at_20(recommend(ti, sessions, popular), tl),
    "history + covisit + popular": recall_at_20(recommend(ti, sessions, popular, covisit=cv), tl),
}

import pandas as pd
pd.DataFrame(results).T.round(4)

,clicks,carts,orders,score
popular only,0.0053,0.0067,0.0060,0.0061
history + popular,0.3259,0.3122,0.5736,0.4704
history + covisit + popular,0.4244,0.3831,0.6170,0.5276
